## 🛠️ Step 1: Clone Repository and Setup Environment


In [ ]:
# Clone the CoFT repository
import os
import subprocess
import sys

# Remove existing directory if it exists
if os.path.exists('/content/CoFT'):
    !rm -rf /content/CoFT
    print("🗑️ Removed existing CoFT directory")

# Clone the repository
print("📥 Cloning CoFT repository...")
!git clone https://github.com/NguyenHuy190303/CoFT.git /content/CoFT

# Change to the project directory
os.chdir('/content/CoFT')
print(f"📁 Changed directory to: {os.getcwd()}")

# List files to verify clone
print("\n📋 Repository contents:")
!ls -la


## ⚙️ Step 2: Run Setup Script


In [ ]:
# Make setup script executable and run it
print("🔧 Making setup script executable...")
!chmod +x setup.sh

print("\n🚀 Running setup script...")
print("⏳ This may take a few minutes to extract data archives...")

# Run setup script
!./setup.sh

print("\n✅ Setup completed!")


## 🎲 Step 2.5: Generate Few-Shot Data Files


In [ ]:
# Generate few-shot training data files (train_1perc.pt, train_5perc.pt)
import torch
import numpy as np
from sklearn.model_selection import train_test_split
import os

def random_sample_with_class_guarantee(X, y, percentage, random_seed=42, max_attempts=50):
    """Random sampling with class guarantee - based on CA-TCC methodology"""
    unique_classes = np.unique(y)
    n_samples = len(y)
    target_size = max(len(unique_classes), int(n_samples * percentage / 100.0))
    
    print(f"🎯 Target: {percentage}% of {n_samples} samples = ~{target_size} samples")
    print(f"📊 Classes required: {len(unique_classes)} classes")
    
    # Multiple attempts with varied seeds
    for attempt in range(max_attempts):
        # Random sampling (no stratify - pure random like original paper)
        _, X_sample, _, y_sample = train_test_split(
            X, y, test_size=target_size, random_state=random_seed + attempt, shuffle=True
        )
        
        # Check if all classes are present
        sample_classes = np.unique(y_sample)
        if len(sample_classes) == len(unique_classes):
            print(f"✅ Success on attempt {attempt + 1}")
            print(f"📈 Final sample size: {len(y_sample)}")
            
            # Print class distribution
            for cls in unique_classes:
                count = np.sum(y_sample == cls)
                original_count = np.sum(y == cls)
                percentage_actual = (count / original_count) * 100
                print(f"   Class {cls}: {count}/{original_count} ({percentage_actual:.1f}%)")
            
            return X_sample, y_sample
        
        # Adaptive target size increase
        target_size = min(target_size + len(unique_classes), n_samples)
    
    # Fallback to stratified if random fails
    print(f"⚠️  Random sampling failed after {max_attempts} attempts")
    print("🔄 Using stratified fallback...")
    
    _, X_sample, _, y_sample = train_test_split(
        X, y, test_size=percentage/100.0, random_state=random_seed, shuffle=True, stratify=y
    )
    
    return X_sample, y_sample


In [ ]:
def generate_few_shot_data():
    """Generate few-shot data files for all datasets"""
    
    datasets = {
        'HAR': 'data/HAR',
        'sleep': 'data/sleep', 
        'epilepsy': 'data/epilepsy',
        'SleepEDF': 'data/SleepEDF'
    }
    
    percentages = [1, 5]
    
    for dataset_name, data_path in datasets.items():
        print(f"\n🔍 Processing {dataset_name} dataset...")
        
        train_file = os.path.join(data_path, 'train.pt')
        
        if not os.path.exists(train_file):
            print(f"❌ {train_file} not found, skipping...")
            continue
            
        # Load original training data
        print(f"📂 Loading {train_file}...")
        train_data = torch.load(train_file, map_location='cpu', weights_only=False)
        
        if isinstance(train_data, dict):
            X = train_data['samples']
            y = train_data['labels']
        else:
            # Handle tuple format (samples, labels)
            X, y = train_data
            
        print(f"📊 Original data: {len(X)} samples, {len(np.unique(y))} classes")
        
        # Convert to numpy for sklearn
        if isinstance(X, torch.Tensor):
            X_np = X.numpy()
        else:
            X_np = np.array(X)
            
        if isinstance(y, torch.Tensor):
            y_np = y.numpy()
        else:
            y_np = np.array(y)
        
        # Generate few-shot datasets
        for perc in percentages:
            print(f"\n🎲 Generating {perc}% subset...")
            
            X_few, y_few = random_sample_with_class_guarantee(
                X_np, y_np, percentage=perc, random_seed=42
            )
            
            # Convert back to tensors
            X_few_tensor = torch.tensor(X_few, dtype=X.dtype)
            y_few_tensor = torch.tensor(y_few, dtype=y.dtype)
            
            # Save few-shot data
            output_file = os.path.join(data_path, f'train_{perc}perc.pt')
            
            few_shot_data = {
                'samples': X_few_tensor,
                'labels': y_few_tensor,
                'metadata': {
                    'percentage': perc,
                    'sampling_method': 'random_with_class_guarantee',
                    'original_samples': len(X),
                    'selected_samples': len(X_few),
                    'random_seed': 42
                }
            }
            
            torch.save(few_shot_data, output_file)
            print(f"💾 Saved: {output_file}")
    
    print("\n🎉 Few-shot data generation completed!")

# Run the generation
print("🚀 Starting few-shot data generation...")
generate_few_shot_data()


## 📦 Step 3: Install Dependencies


In [ ]:
# Fix NumPy compatibility issue first
print("🔧 Fixing NumPy compatibility...")
!pip install "numpy<2.0" --force-reinstall

# Install required packages with compatible versions
print("\n📦 Installing dependencies with NumPy 1.x compatibility...")
!pip install -r requirements.txt

# Install additional packages that are compatible with NumPy 1.x
print("\n🔧 Installing Colab-specific packages...")
!pip install "scikit-learn>=1.3.0,<1.4.0" "numpy>=1.21.0,<2.0" torch torchvision torchaudio

# Verify NumPy version
print("\n🔍 Verifying NumPy installation...")
import numpy as np
print(f"✅ NumPy version: {np.__version__}")

print("\n✅ All dependencies installed with NumPy 1.x compatibility!")


## 🔍 Step 4: Verify Setup and Run Examples


In [ ]:
# Verify data extraction
print("🔍 Verifying data extraction...")
data_dirs = ['data/HAR', 'data/sleep', 'data/epilepsy', 'data/SleepEDF']

for data_dir in data_dirs:
    if os.path.exists(data_dir):
        file_count = len([f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))])
        print(f"✅ {data_dir}: {file_count} files extracted")
    else:
        print(f"❌ {data_dir}: Directory not found")

# Verify Python imports
print("\n🐍 Verifying Python imports...")
try:
    import torch
    import numpy as np
    import sklearn
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"✅ NumPy: {np.__version__}")
    print(f"✅ Scikit-learn: {sklearn.__version__}")
    print(f"✅ CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
except ImportError as e:
    print(f"❌ Import error: {e}")

print("\n🎉 Setup verification completed!")


In [ ]:
!python3 main.py --training_mode full_run --selected_dataset sleep --enable_coft --label_percentage 1

In [ ]:
!python3 main.py --training_mode full_run --selected_dataset sleep --enable_coft --label_percentage 5